# Checking fault equivalence

### Setup

In [ ]:
import math
from typing import List
from pyzx.graph.graph_s import GraphS

def _organize_in_ring(g: GraphS, nodes: List[int], radius: float) -> None:
    n = len(nodes)
    step = 2 * math.pi / n  # angular spacing

    for i, node in enumerate(nodes):
        g.set_qubit(node, radius * math.sin(i * step))
        g.set_row(node, radius * math.cos(i * step))

def small_ring(ring_size: int) -> GraphS:
    g = GraphS()
    b_spiders_2 = [g.add_vertex(zx.VertexType.BOUNDARY) for _ in range(ring_size)]
    z = g.add_vertex(zx.VertexType.Z, qubit=0, row=0)
    for i in range(ring_size):
        g.add_edge((z, b_spiders_2[i]))
    _organize_in_ring(g, b_spiders_2, radius=3.0)

    return g

def big_ring(ring_size: int) -> GraphS:
    g = GraphS()
    b_spiders = [g.add_vertex(zx.VertexType.BOUNDARY) for _ in range(ring_size)]
    z_spiders = [g.add_vertex(zx.VertexType.Z) for _ in range(ring_size)]
    for i in range(ring_size):
        g.add_edge((z_spiders[i-1], z_spiders[i]))
        g.add_edge((z_spiders[i], b_spiders[i]))
    _organize_in_ring(g, b_spiders, radius=5.0)
    _organize_in_ring(g, z_spiders, radius=3.0)

    return g

### Collapsing a ring of five Z-spiders is fault equivalent

In [ ]:
import pyzx as zx

g1 = big_ring(ring_size=5)
g2 = small_ring(ring_size=5)
zx.draw(g1, labels=True)
zx.draw(g2, labels=True)

In [ ]:
from faultgadget.query.equivalence import is_fault_equivalent

is_fault_equivalent(g1, g2, quiet=True)

### But collapsing a ring of six Z-spiders is NOT fault equivalent

In [ ]:
g1 = big_ring(ring_size=6)
g2 = small_ring(ring_size=6)
zx.draw(g1, labels=True)
zx.draw(g2, labels=True)

In [ ]:
from faultgadget.query.equivalence import is_fault_equivalent

is_fault_equivalent(g1, g2, quiet=False) # You can also inspect what is happening under the hood / what goes wrong with quiet=False